In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from becsim import (SimulationConfig, Grid, make_cutoff, make_terms,
                    make_potential, RealTimeEvolution, ImaginaryTimeEvolution,
                    Recorder, NormMonitor, storage, plotting)

device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = SimulationConfig(
    Np=1e5, m=1.9e-25, omega=2 * torch.pi * 40,
    a0=2e-9, a02=5e-23,
    up=2.5e-5, N=256,
    sigma=10e-6, ev_time=0.015, time_steps=10_000,
    modulation_amp=0.1, modulation_freq=2 * 5**0.5,
    cutoff="Hard Cutoff", cutoff_coeff=0.9,
)

In [ ]:
grid   = Grid(cfg, device)
cutoff = make_cutoff(cfg, grid)
terms  = make_terms(cfg, grid, cutoff)

# --- ground state, from cache if the physics matches ---
path = storage.ground_state_path(cfg)
if path.exists():
    psi, meta = storage.load_ground_state(path, cfg, device)
    print(f"Loaded ground state from {path} (E = {meta['final_energy']:.6f})")
else:
    rec = Recorder([t.name for t in terms], track_modes=False)
    ite = ImaginaryTimeEvolution(
        grid, make_potential(cfg, grid, cfg.imag_gamma), terms,
        dtau=cfg.imag_dtau, max_steps=cfg.imag_max_steps,
        recorder=rec, ke_divisor=cfg.N**3, tolerance=cfg.tolerance)
    psi = ite.run(grid.gaussian().to(device))
    storage.save_ground_state(path, psi, cfg, converged=ite.converged,
                              final_energy=rec.last_total(),
                              steps=ite.steps_taken)

In [ ]:
rec = Recorder([t.name for t in terms],
               radius_squared=grid.radius_squared(), dV=grid.dV) # UPDATE INPUTS

rte = RealTimeEvolution(
    grid, make_potential(cfg, grid, cfg.gamma), terms,
    dtau=cfg.dtau, max_steps=cfg.time_steps,
    recorder=rec, ke_divisor=cfg.N**3,
    monitors=[NormMonitor(grid)])

psi_final = rte.run(psi)
storage.save_run("storage/run_2026_09_07", cfg, rec,
                 psi_final=psi_final.cpu().numpy())

In [ ]:
cfg_dict, arrays = storage.load_run("storage/run_2026_09_07/run.npz")

fig = plotting.energy_history(times, arrays["energies"], list(arrays["columns"]))
fig.axes[0].set_yscale("log")          # the thing you could not do before
fig

In [ ]:
from scipy import constants
import math
pi = constants.pi
simulation_parameters = {"Np":1e5, 
                         "m":1.9e-25, 
                         "omega":2*pi*40, 
                         "a0": 2e-9, 
                         "a02": 5e-23,#(2/3)*1e-27, 
                         "gamma": 1, 
                         "imag_gamma": 1,#math.sqrt(2),
                         "modulation freq": 0, #None, #unitless freq
                         "modulation amplitude": 0.0, #10% of potential
                         "sigma": 10e-6, 
                         "up": 2.5e-5,
                         "N": 256, 
                         "ev_time": 0.015,
                         "time_steps": 10000,
                         "cutoff": "Hard Cutoff",
                         "cutoff_coeff": 0.9, 
                         "include_g0": True,
                         "waist": True,
                         "high_precision": True, 
                         "z_scale": 1, 
                         "y_scale": 1,
                         "time_plots": True, 
                         "final_plots": True, 
                         "plot_steps": 0.01, 
                         "Nz": None,
                         "imag_dtau": 1e-4, 
                         "imag_max_steps": 10000, 
                         "tolerance": 1e-6,
                         "init_type": "Ground State",
                         "device": "cpu"}
baseline_params = dict(simulation_parameters)
baseline_params.update(N=32, up=7.5e-6, time_steps=200, ev_time=3e-4,
                       init_type="Gaussian", time_plots=False,
                       high_precision=True)
print(baseline_params)

{'Np': 100000.0, 'm': 1.9e-25, 'omega': 251.32741228718345, 'a0': 2e-09, 'a02': 5e-23, 'gamma': 1, 'imag_gamma': 1, 'modulation freq': 4.47213595499958, 'modulation amplitude': 0.1, 'sigma': 1e-05, 'up': 7.5e-06, 'N': 32, 'ev_time': 0.0003, 'time_steps': 200, 'cutoff': 'Hard Cutoff', 'cutoff_coeff': 0.9, 'include_g0': True, 'waist': True, 'high_precision': True, 'z_scale': 1, 'y_scale': 1, 'time_plots': False, 'final_plots': True, 'plot_steps': 0.01, 'Nz': None, 'imag_dtau': 0.0001, 'imag_max_steps': 10000, 'tolerance': 1e-06, 'init_type': 'Gaussian', 'device': 'cpu'}


In [27]:
%load_ext autoreload
%autoreload 2

import torch
from config import SimulationConfig 
from grid import Grid
from cutoff import make_cutoff 
from interactions import make_terms
from potentials import make_potential 
from evolution import RealTimeEvolution, ImaginaryTimeEvolution
from diagnostics import Recorder, NormMonitor
import storage, plotting

device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = SimulationConfig(
    Np=1e5, m=1.9e-25, omega=2 * torch.pi * 40,
    a0=2e-9, a02=5e-23,
    up=7.5e-6, N=32,
    sigma=10e-6, ev_time=3e-4, time_steps=200,
    cutoff="Hard Cutoff", cutoff_coeff=0.9,
)

grid   = Grid(cfg, device)
cutoff = make_cutoff(cfg, grid)
terms  = make_terms(cfg, grid, cutoff)

rec = Recorder([t.name for t in terms], measure_every=10, track_modes=cfg.track_modes, dt = cfg.dtau, radius_squared=grid.radius_squared() if cfg.track_waist else None, dV=grid.dV)

rte = RealTimeEvolution(
    grid, make_potential(cfg, grid, cfg.gamma), terms,
    dtau=cfg.dtau, max_steps=cfg.time_steps,
    recorder=rec,
    monitors=[NormMonitor(grid)])

psi_final = rte.run(grid.gaussian().to(device))
storage.save_run("storage/run_2026_09_07", cfg, grid,
                 rec, psi_final=psi_final.cpu().numpy())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


PosixPath('storage/run_2026_09_07/run.npz')

In [35]:
import numpy as np

base = np.load("../baseline_small.npz")

def compare(name, new, old, rtol=1e-12, atol=None):
    new, old = np.asarray(new), np.asarray(old)
    if atol is None:
        atol = 1e-15 * np.max(np.abs(old))   # floor relative to the array's scale
    ok = np.allclose(new, old, rtol=rtol, atol=atol)
    signif = np.abs(old) > atol              # ignore entries that are numerically zero
    worst = (np.max(np.abs(new[signif] - old[signif]) / np.abs(old[signif]))
             if signif.any() else 0.0)
    print(f"{name:12s} {'PASS' if ok else 'FAIL'}   worst rel diff = {worst:.3e} "
          f"(atol={atol:.2e}, {(~signif).sum()} near-zero entries skipped)")
    return ok

compare("psi_final", psi_final.cpu().numpy(), base["psi_final"])
compare("energies",  rec.energies(),        base["energies"])
#compare("kx",        rec.modes()[0],        base["kx"])

psi_final    PASS   worst rel diff = 1.851e-14 (atol=4.06e-17, 0 near-zero entries skipped)
energies     FAIL   worst rel diff = 7.635e-03 (atol=1.18e-14, 1 near-zero entries skipped)


False

In [34]:
base["energies"] - rec.energies()

array([[ 3.53452067e-07,  0.00000000e+00, -1.11022302e-16,
        -8.84674303e-20,  3.53452066e-07],
       [ 1.66553129e-06,  3.55271368e-15,  0.00000000e+00,
        -6.80516320e-21,  1.66553129e-06],
       [ 2.97760468e-06,  0.00000000e+00,  0.00000000e+00,
        -1.02077810e-19,  2.97760468e-06],
       [ 4.28966681e-06,  0.00000000e+00, -1.11022302e-16,
         2.44986496e-19,  4.28966681e-06],
       [ 5.60171225e-06,  1.77635684e-15, -1.11022302e-16,
         2.51792124e-19,  5.60171225e-06],
       [ 6.91373556e-06, -1.77635684e-15,  1.11022302e-16,
         3.94700810e-19,  6.91373555e-06],
       [ 8.22573131e-06, -1.77635684e-15,  0.00000000e+00,
         7.89401620e-19,  8.22573130e-06],
       [ 9.53769407e-06,  0.00000000e+00,  0.00000000e+00,
         1.31340069e-18,  9.53769407e-06],
       [ 1.08496184e-05,  1.77635684e-15, -2.22044605e-16,
         1.48353189e-18,  1.08496184e-05],
       [ 1.21614990e-05, -1.77635684e-15, -2.22044605e-16,
         1.59241201e-18